<a href="https://colab.research.google.com/github/lunecarvalho/newslens-development/blob/main/02_topic_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('/content/cleaned_dataset.csv')

print(f'Total de registros: {len(df)}')
df.head()

Total de registros: 7199


,id,label,arquivo,texto_original,texto_preprocessado,tema
0,1,falso,1170.txt,Falso médico é preso pela PM de Santa Catarina...,falso médico é preso pela pm de santa catarina...,NaN
1,2,falso,1688.txt,Antes da confirmação da morte de Teori Zavasck...,antes da confirmação da morte de teori zavasck...,NaN
2,3,falso,1863.txt,Senador que assume o lugar de Renan foi flagra...,senador que assume o lugar de renan foi flagra...,NaN
3,4,falso,1563.txt,"Repórter relata conversa entre deputados: ""A D...",repórter relata conversa entre deputados: a di...,NaN
4,5,falso,3556.txt,Governo repressor e comunista usa exército par...,governo repressor e comunista usa exército par...,NaN


In [5]:
print('Textos vazios:')
print((df['texto_preprocessado'].fillna('').str.strip() == '').sum())

Textos vazios:
0


In [6]:
df['tamanho_texto'] = df['texto_preprocessado'].str.len()

df['tamanho_texto'].describe()

,tamanho_texto
count,7199.000000
mean,1103.320878
std,697.240864
min,46.000000
25%,694.000000
50%,946.000000
75%,1340.000000
max,12472.000000


In [7]:
df.drop(columns=['tamanho_texto'], inplace=True)

In [8]:
print('Textos duplicados:', df['texto_preprocessado'].duplicated().sum())

Textos duplicados: 0


In [9]:
!pip install bertopic

In [10]:
documentos = df['texto_preprocessado'].fillna('').tolist()

print(f'Total de documentos: {len(documentos)}')

Total de documentos: 7199


In [11]:
print(documentos[0])

falso médico é preso pela pm de santa catarina: sou formado em medicina pelo seriado greys anatomy. josias de farias júnior se passava por médico em um hospital da unimed em balneário camboriú, santa catarina. o estelionatário de 19 anos foi preso pela polícia na noite da última terça-feira (30). ele portava prontuários, carimbos médicos, um jaleco e um estetoscópio que, de acordo com testemunhas, foram roubados de um médico do próprio hospital. seguranças do local desconfiaram da credencial do jovem e o detiveram até a chegada da polícia. ainda não se sabe se ele chegou a atender algum paciente ou até mesmo prescreveu alguma receita. durante o depoimento, ele afirmou ser fã de um seriado de tv (sobre medicina) muito famoso nos eua: greys anatomy. em um vídeo postado nas redes sociais ( minuto 0:52 ) josias declara ser um fã apaixonado das séries greys anatomy e bones.


In [12]:
for i in range(3):
    print(f'\nDOCUMENTO {i + 1}:')
    print(documentos[i][:500])


DOCUMENTO 1:
falso médico é preso pela pm de santa catarina: sou formado em medicina pelo seriado greys anatomy. josias de farias júnior se passava por médico em um hospital da unimed em balneário camboriú, santa catarina. o estelionatário de 19 anos foi preso pela polícia na noite da última terça-feira (30). ele portava prontuários, carimbos médicos, um jaleco e um estetoscópio que, de acordo com testemunhas, foram roubados de um médico do próprio hospital. seguranças do local desconfiaram da credencial do 

DOCUMENTO 2:
antes da confirmação da morte de teori zavascki, senador postou mensagem estranha no twitter. senador disse que o jornal nacional iria lançar uma bomba sobre o brasil. será uma infeliz coincidência? uma mensagem enigmática foi postada pelo senador josé medeiros (psdmt) pouco antes da confirmação da morte do ministro teori zavascki: não vou antecipar furo porque não sou jornalista mas o jornal nacional hj trará uma bomba de forte impacto no brasil, envolvendo stf. ( 1

In [13]:
!pip install sentence-transformers

In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    'paraphrase-multilingual-MiniLM-L12-v2'
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [15]:
embeddings = embedding_model.encode(
    documentos,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/225 [00:00<?, ?it/s]

(7199, 384)


In [16]:
!pip install bertopic umap-learn hdbscan

In [17]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

In [18]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42
)

In [19]:
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

In [20]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    max_df=1.0
)

In [21]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language='multilingual',
    verbose=True
)

In [22]:
topics, probabilities = topic_model.fit_transform(
    documentos,
    embeddings
)

2026-08-27 12:10:46,927 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-27 12:11:54,901 - BERTopic - Dimensionality - Completed ✓
2026-08-27 12:11:54,904 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-27 12:11:55,234 - BERTopic - Cluster - Completed ✓
2026-08-27 12:11:55,241 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-27 12:12:02,081 - BERTopic - Representation - Completed ✓


In [23]:
topic_info = topic_model.get_topic_info()

topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2647,-1_de_que_do_da,"[de, que, do, da, em, para, no, um, na, com]",[atualizada às 12h07 o senador delcídio amaral...
1,0,363,0_brasil_que_de_do,"[brasil, que, de, do, um, da, em, não, para, no]",[lula nunca foi modesto. já se comparou a jesu...
2,1,346,1_norte_coreia_do norte_coreia do,"[norte, coreia, do norte, coreia do, kim, eua,...",[coreia do norte lança míssil no litoral japon...
3,2,266,2_de_para_bilhões_em,"[de, para, bilhões, em, do, que, da, com, no, ...",[veio para confundir incerteza sobre reais int...
4,3,261,3_milhões_odebrecht_de_da,"[milhões, odebrecht, de, da, que, em, dinheiro...",[baiano entrega documentos à pf que comprovam ...
...,...,...,...,...,...
70,69,16,69_internet_banda larga_larga_hackers,"[internet, banda larga, larga, hackers, que, d...",[internet limitada: 12 horas de netflix ou alg...
71,70,15,70_aplicativos_uber_caminhoneiros_transporte,"[aplicativos, uber, caminhoneiros, transporte,...",[prefeitura de sp flexibiliza futuras regras p...
72,71,15,71_pf_da pf_diretor geral_daiello,"[pf, da pf, diretor geral, daiello, diretor, n...",[delegado fernando segóvia assumirá comando da...
73,72,15,72_prefeito_doria_vereadores_crivella,"[prefeito, doria, vereadores, crivella, promes...",[doria deixa promessas de campanha de fora do ...


In [24]:
print(f'Quantidade de tópicos: {len(topic_info) - 1}')

Quantidade de tópicos: 74


In [25]:
topic_info.head(15)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2647,-1_de_que_do_da,"[de, que, do, da, em, para, no, um, na, com]",[atualizada às 12h07 o senador delcídio amaral...
1,0,363,0_brasil_que_de_do,"[brasil, que, de, do, um, da, em, não, para, no]",[lula nunca foi modesto. já se comparou a jesu...
2,1,346,1_norte_coreia_do norte_coreia do,"[norte, coreia, do norte, coreia do, kim, eua,...",[coreia do norte lança míssil no litoral japon...
3,2,266,2_de_para_bilhões_em,"[de, para, bilhões, em, do, que, da, com, no, ...",[veio para confundir incerteza sobre reais int...
4,3,261,3_milhões_odebrecht_de_da,"[milhões, odebrecht, de, da, que, em, dinheiro...",[baiano entrega documentos à pf que comprovam ...
5,4,203,4_dilma_impeachment_presidente_que,"[dilma, impeachment, presidente, que, do, de, ...",[manter o discurso de golpe é impróprio ao paí...
6,5,188,5_stf_supremo_ministro_do,"[stf, supremo, ministro, do, mendes, gilmar, t...",[fachin envia à primeira instância denúncia co...
7,6,181,6_lula_ex_ex presidente_moro,"[lula, ex, ex presidente, moro, da, do, de, de...",[condenação de lula na 2ª instância: veja as p...
8,7,146,7_candidato_alckmin_partido_psdb,"[candidato, alckmin, partido, psdb, bolsonaro,...","[em evento com alckmin e doria, fhc defende pr..."
9,8,138,8_igreja_pastor_que_de,"[igreja, pastor, que, de, um, uma, deus, da ig...",[pastor que anda de ferrari se pronuncia: o ca...


In [26]:
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [27]:
from nltk.corpus import stopwords

stopwords_pt = stopwords.words('portuguese')

print(stopwords_pt[:20])

['a', 'à', 'ao', 'aos', 'aquela', 'aquelas', 'aquele', 'aqueles', 'aquilo', 'as', 'às', 'até', 'com', 'como', 'da', 'das', 'de', 'dela', 'delas', 'dele']


In [28]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer_model = CountVectorizer(
    stop_words=stopwords_pt,
    ngram_range=(1, 2),
    min_df=1
)

In [29]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language='multilingual',
    verbose=True
)

In [30]:
topics, probabilities = topic_model.fit_transform(
    documentos,
    embeddings
)

2026-08-27 12:12:04,888 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-27 12:12:23,429 - BERTopic - Dimensionality - Completed ✓
2026-08-27 12:12:23,430 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-27 12:12:24,017 - BERTopic - Cluster - Completed ✓
2026-08-27 12:12:24,024 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-27 12:12:29,946 - BERTopic - Representation - Completed ✓


In [31]:
topic_info = topic_model.get_topic_info()

topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2647,-1_lula_presidente_ex_federal,"[lula, presidente, ex, federal, ex presidente,...",[atualizada às 12h07 o senador delcídio amaral...
1,0,363,0_brasil_país_presidente_lula,"[brasil, país, presidente, lula, brasileiros, ...",[a defesa de aécio neves exigiu que o stf afas...
2,1,346,1_norte_coreia_coreia norte_kim,"[norte, coreia, coreia norte, kim, eua, nuclea...",[coreia do norte alerta países: não se juntem ...
3,2,266,2_bilhões_governo_ano_milhões,"[bilhões, governo, ano, milhões, 2017, trabalh...",[saiba como foi a semana na economia. redução ...
4,3,261,3_odebrecht_milhões_dinheiro_delação,"[odebrecht, milhões, dinheiro, delação, ex, pr...",[delação de ex-diretor da odebrecht atinge tem...
...,...,...,...,...,...
70,69,16,69_internet_banda larga_larga_hackers,"[internet, banda larga, larga, hackers, banda,...",[internet limitada: 12 horas de netflix ou alg...
71,70,15,70_aplicativos_uber_caminhoneiros_transporte,"[aplicativos, uber, caminhoneiros, transporte,...",[prefeitura de sp flexibiliza futuras regras p...
72,71,15,71_pf_diretor geral_daiello_diretor,"[pf, diretor geral, daiello, diretor, novo dir...",[delegado fernando segóvia assumirá comando da...
73,72,15,72_prefeito_doria_vereadores_crivella,"[prefeito, doria, vereadores, crivella, promes...",[doria deixa promessas de campanha de fora do ...


In [32]:
print(topic_model.get_topic_info()[['Topic', 'Count', 'Name']].to_string(index=False))

 Topic  Count                                            Name
    -1   2647                   -1_lula_presidente_ex_federal
     0    363                   0_brasil_país_presidente_lula
     1    346                 1_norte_coreia_coreia norte_kim
     2    266                   2_bilhões_governo_ano_milhões
     3    261            3_odebrecht_milhões_dinheiro_delação
     4    203         4_dilma_impeachment_presidente_rousseff
     5    188                   5_stf_supremo_ministro_mendes
     6    181                    6_lula_ex presidente_moro_ex
     7    146                7_candidato_alckmin_partido_psdb
     8    138                       8_igreja_pastor_deus_papa
     9    137                           9_anos_mãe_jovem_bebê
    10    107               10_avião_aeronave_acidente_piloto
    11    104                           11_prisão_preso_pf_ex
    12     94           12_manifestantes_ato_protesto_polícia
    13     79                        13_lula_dilma_petista_pt
    14  

In [33]:
print("Quantidade de tópicos:", len(topic_model.get_topic_info()) - 1)

Quantidade de tópicos: 74


In [34]:
topic_model.get_topic(0)

[('brasil', np.float64(0.013682024197759761)),
 ('país', np.float64(0.007358838691703302)),
 ('presidente', np.float64(0.006870159021953427)),
 ('lula', np.float64(0.0056173653802873745)),
 ('brasileiros', np.float64(0.005185127281866319)),
 ('brasileiro', np.float64(0.004910664251309181)),
 ('temer', np.float64(0.004898818470295437)),
 ('política', np.float64(0.00470586834950469)),
 ('governo', np.float64(0.00449558909555325)),
 ('brasileira', np.float64(0.004402832896259538))]

In [35]:
topic_model.get_topic(1)

[('norte', np.float64(0.03685486679427854)),
 ('coreia', np.float64(0.03114385811602478)),
 ('coreia norte', np.float64(0.02334043602889135)),
 ('kim', np.float64(0.02001704431130594)),
 ('eua', np.float64(0.019487603839474326)),
 ('nuclear', np.float64(0.015916850518809658)),
 ('un', np.float64(0.015397884620307289)),
 ('trump', np.float64(0.015143009660840973)),
 ('coréia', np.float64(0.014152831766273345)),
 ('sul', np.float64(0.013613721947014122))]

In [36]:
topic_info = topic_model.get_topic_info()

print(
    topic_info[['Topic', 'Count', 'Name']]
    .to_string(index=False)
)

 Topic  Count                                            Name
    -1   2647                   -1_lula_presidente_ex_federal
     0    363                   0_brasil_país_presidente_lula
     1    346                 1_norte_coreia_coreia norte_kim
     2    266                   2_bilhões_governo_ano_milhões
     3    261            3_odebrecht_milhões_dinheiro_delação
     4    203         4_dilma_impeachment_presidente_rousseff
     5    188                   5_stf_supremo_ministro_mendes
     6    181                    6_lula_ex presidente_moro_ex
     7    146                7_candidato_alckmin_partido_psdb
     8    138                       8_igreja_pastor_deus_papa
     9    137                           9_anos_mãe_jovem_bebê
    10    107               10_avião_aeronave_acidente_piloto
    11    104                           11_prisão_preso_pf_ex
    12     94           12_manifestantes_ato_protesto_polícia
    13     79                        13_lula_dilma_petista_pt
    14  

In [37]:
topic_model.visualize_topics()

In [38]:
topic_model.visualize_barchart(top_n_topics=20)

In [42]:
topic_model.get_representative_docs(0)

['a defesa de aécio neves exigiu que o stf afaste edson fachin da relatoria do processo que o afasta do senado. nunca o samba do crioulo doido foi tão atual como no brasil hoje, como lembra eliane cantanhêde na coluna no estadão. nesta nau dos insensatos, a defesa escolhe o juiz de acordo com as conveniências do réu e o fato de ele já ter delinquido em outra ocasião o autoriza a fazê-lo. quanto mais diversas forem as causas contra o acusado mais ele se acha no direito de se livrar de quem o condenou antes e procurar quem tradicionalmente o liberta, caso dos tucanos que sempre buscam e conseguem cair nas mãos generosas de gilmar mendes e alexandre de moraes e longe de juízes mais rigorosos como fachin. (comentário no jornal eldorado da rádio eldorado fm 107,3 na terça-feira 3 de outubro de 2017, às 7h30m) para ouvir clique aqui e, em seguida, no play para ouvir o samba do crioulo doido com os demônios da garoa clique aqui abaixo, a degravação do comentário na íntegra: eldorado, 3 de out

In [40]:
topic_model.get_representative_docs(24)

['dezessete países registram casos de microcefalia aossociada ao zika. vírus já tem transmissão em 70 países, conforme dados da oms desde 2007; nos estados unidos, autoridades acreditam que surto deve crescer. são paulo - depois de meses de dúvidas sobre a ligação entre vírus e má-formação, e um alerta mundial, 17 países já registram casos de microcefalia associada ao zika, de acordo com o boletim mais recente da organização mundial de saúde (oms). em quatro, não há transmissão local do vírus, mas as mães viajaram a locais onde há epidemia. desde 2007, 70 países tiveram transmissão de zika. o balanço é de 1.926 casos de má-formação no mundo desde 2014. o brasil ainda concentra a maior parte dos registros associados ao zika (1.835). outros 2.957 casos no país seguem em investigação pelo ministério da saúde em 4.223, a relação com vírus e com problemas congênitos foi descartada. na sequência, têm mais notificações de microcefalia colômbia (24) e estados unidos (21). dos registros america

In [43]:
outliers = [
    documentos[i]
    for i, topic in enumerate(topics)
    if topic == -1
]

print(f'Total de outliers: {len(outliers)}')

Total de outliers: 2647


In [44]:
for i, texto in enumerate(outliers[:10]):
    print(f'\n--- OUTLIER {i+1} ---')
    print(texto[:1000])


--- OUTLIER 1 ---
tico santa cruz, o papagaio de pirata do pt, tenta defender dilma e leva porretada de internauta. tico santa cruz, acostumado a discursar para um bando de petistas alienados, encontrou pela frente uma parada dura!. o flagra foi publicado pelo site bolsonaro opressor o músico publicou (nas redes sociais) uma imagem com algumas manchetes dos principais jornais do país em que a pauta era pedalar não é crime . tico tentou induzir seus seguidores a imaginar que dilma não havia cometido crime algum. durante a última semana, o mpf concluiu que as denominadas pedaladas fiscais não configuram crimes comuns. vamos repetir: as pedaladas ficais (de acordo como mpf) não configuram crimes comuns ! vamos repetir novamente: crimes comuns! o detalhe é que o notável músico não se deu ao trabalho ler as notícias (na íntegra) citadas em sua postagens. um seguidor da página argumentou: o artista respondeu imediatamente: eis que veio a paulada do internauta:

--- OUTLIER 2 ---
alemanha vo

In [45]:
topic_info = topic_model.get_topic_info()

topicos = topic_info[
    topic_info['Topic'] != -1
][['Topic', 'Count', 'Name']]

print(topicos.to_string(index=False))

 Topic  Count                                            Name
     0    363                   0_brasil_país_presidente_lula
     1    346                 1_norte_coreia_coreia norte_kim
     2    266                   2_bilhões_governo_ano_milhões
     3    261            3_odebrecht_milhões_dinheiro_delação
     4    203         4_dilma_impeachment_presidente_rousseff
     5    188                   5_stf_supremo_ministro_mendes
     6    181                    6_lula_ex presidente_moro_ex
     7    146                7_candidato_alckmin_partido_psdb
     8    138                       8_igreja_pastor_deus_papa
     9    137                           9_anos_mãe_jovem_bebê
    10    107               10_avião_aeronave_acidente_piloto
    11    104                           11_prisão_preso_pf_ex
    12     94           12_manifestantes_ato_protesto_polícia
    13     79                        13_lula_dilma_petista_pt
    14     78                       14_anos_morreu_ator_corpo
    15  

In [46]:
df['topico_bertopic'] = topics

In [47]:
df[['id', 'arquivo', 'topico_bertopic', 'texto_original']].head()

,id,arquivo,topico_bertopic,texto_original
0,1,1170.txt,28,Falso médico é preso pela PM de Santa Catarina...
1,2,1688.txt,0,Antes da confirmação da morte de Teori Zavasck...
2,3,1863.txt,55,Senador que assume o lugar de Renan foi flagra...
3,4,1563.txt,3,"Repórter relata conversa entre deputados: ""A D..."
4,5,3556.txt,12,Governo repressor e comunista usa exército par...


In [48]:
topic_info = topic_model.get_topic_info()

topic_info = topic_info[
    topic_info['Topic'] != -1
][['Topic', 'Count', 'Name']]

topic_info

,Topic,Count,Name
1,0,363,0_brasil_país_presidente_lula
2,1,346,1_norte_coreia_coreia norte_kim
3,2,266,2_bilhões_governo_ano_milhões
4,3,261,3_odebrecht_milhões_dinheiro_delação
5,4,203,4_dilma_impeachment_presidente_rousseff
...,...,...,...
70,69,16,69_internet_banda larga_larga_hackers
71,70,15,70_aplicativos_uber_caminhoneiros_transporte
72,71,15,71_pf_diretor geral_daiello_diretor
73,72,15,72_prefeito_doria_vereadores_crivella


In [49]:
topic_info['tema'] = ''

In [81]:
mapa_temas = {
    0: 'Política',
    1: 'Internacional',
    2: 'Economia',
    3: 'Corrupção',
    4: 'Política',
    5: 'Política',
    6: 'Política',
    7: 'Política',
    8: 'Religião',
    9: 'Sociedade',
    10: 'Acidentes',
    11: 'Segurança',
    12: 'Política',
    13: 'Política',
    14: 'Entretenimento',
    15: 'Entretenimento',
    16: 'Política',
    17: 'Segurança',
    18: 'Segurança',
    19: 'Política',
    20: 'Internacional',
    21: 'Segurança',
    22: 'Corrupção',
    23: 'Meio Ambiente',
    24: 'Saúde',
    25: 'Política',
    26: 'Política',
    27: 'Cultura',
    28: 'Saúde',
    29: 'Cultura',
    30: 'Ciência',
    31: 'Política',
    32: 'Política',
    33: 'Esportes',
    34: 'Política',
    35: 'Alimentação',
    36: 'Política',
    37: 'Esportes',
    38: 'Entretenimento',
    39: 'Cultura',
    40: 'Política',
    41: 'Política',
    42: 'Saúde',
    43: 'Mídia',
    44: 'Segurança',
    45: 'Tecnologia',
    46: 'Saúde',
    47: 'Política',
    48: 'Internacional',
    49: 'Acidentes',
    50: 'Internacional',
    51: 'Política',
    52: 'Segurança',
    53: 'Internacional',
    54: 'Sociedade',
    55: 'Política',
    56: 'Política',
    57: 'Política',
    58: 'Educação',
    59: 'Economia',
    60: 'Política',
    61: 'Corrupção',
    62: 'Mídia',
    63: 'Política',
    64: 'Política',
    65: 'Política',
    66: 'Animais',
    67: 'Internacional',
    68: 'Saúde',
    69: 'Tecnologia',
    70: 'Transportes',
    71: 'Política',
    72: 'Política',
    73: 'Economia'
}

In [82]:
topic_info['tema'] = topic_info['Topic'].map(mapa_temas)

In [83]:
topic_info[['Topic', 'Count', 'Name', 'tema']]

,Topic,Count,Name,tema
1,0,363,0_brasil_país_presidente_lula,Política
2,1,346,1_norte_coreia_coreia norte_kim,Internacional
3,2,266,2_bilhões_governo_ano_milhões,Economia
4,3,261,3_odebrecht_milhões_dinheiro_delação,Corrupção
5,4,203,4_dilma_impeachment_presidente_rousseff,Política
...,...,...,...,...
70,69,16,69_internet_banda larga_larga_hackers,Tecnologia
71,70,15,70_aplicativos_uber_caminhoneiros_transporte,Transportes
72,71,15,71_pf_diretor geral_daiello_diretor,Política
73,72,15,72_prefeito_doria_vereadores_crivella,Política


In [84]:
for topic in [0, 2, 3, 9, 14, 23, 28, 35, 36, 43, 46, 54, 63, 68, 70, 73]:
    print(f'\n========== TÓPICO {topic} ==========')
    print(topic_model.get_topic(topic)[:10])


========== TÓPICO 0 ==========
[('brasil', np.float64(0.013682024197759761)), ('país', np.float64(0.007358838691703302)), ('presidente', np.float64(0.006870159021953427)), ('lula', np.float64(0.0056173653802873745)), ('brasileiros', np.float64(0.005185127281866319)), ('brasileiro', np.float64(0.004910664251309181)), ('temer', np.float64(0.004898818470295437)), ('política', np.float64(0.00470586834950469)), ('governo', np.float64(0.00449558909555325)), ('brasileira', np.float64(0.004402832896259538))]

========== TÓPICO 2 ==========
[('bilhões', np.float64(0.012547523737331073)), ('governo', np.float64(0.009005187282879464)), ('ano', np.float64(0.008536010733366788)), ('milhões', np.float64(0.005864468657362612)), ('2017', np.float64(0.005831774431370531)), ('trabalho', np.float64(0.005687387535212013)), ('feira', np.float64(0.005480898533781865)), ('mil', np.float64(0.005474943437246456)), ('mercado', np.float64(0.005249460718951768)), ('empresas', np.float64(0.005246147807387492))]



In [85]:
for topic in [0, 2, 3, 9, 14, 23, 28, 35, 36, 43, 46, 54, 63, 68, 70, 73]:
    print(f'\n========== TÓPICO {topic} ==========')
    docs = topic_model.get_representative_docs(topic)

    for i, doc in enumerate(docs[:2]):
        print(f'\nDocumento {i+1}:')
        print(doc[:500])


========== TÓPICO 0 ==========

Documento 1:
a defesa de aécio neves exigiu que o stf afaste edson fachin da relatoria do processo que o afasta do senado. nunca o samba do crioulo doido foi tão atual como no brasil hoje, como lembra eliane cantanhêde na coluna no estadão. nesta nau dos insensatos, a defesa escolhe o juiz de acordo com as conveniências do réu e o fato de ele já ter delinquido em outra ocasião o autoriza a fazê-lo. quanto mais diversas forem as causas contra o acusado mais ele se acha no direito de se livrar de quem o conden

Documento 2:
temer pode entregar o poder aos militares após renuncia. temer está em constante contato com os comandantes das forças armadas nos últimos dias. durante as manifestações em brasília os atos de vandalismos mostraram ao brasil o que podem vir a ser os dias após o impeachment. acossado por denúncias de delatores da jf na lava jato, o presidente michel temer manteve uma reunião com o ministro da defesa, raul jungmann, e os três comandantes

In [86]:
dados_topicos = []

for topic in topic_info['Topic']:
    palavras = topic_model.get_topic(topic)

    palavras_str = ', '.join([
        palavra for palavra, peso in palavras[:10]
    ])

    docs = topic_model.get_representative_docs(topic)

    exemplo = docs[0][:500] if docs else ''

    dados_topicos.append({
        'topico': topic,
        'palavras': palavras_str,
        'exemplo': exemplo
    })

df_topicos = pd.DataFrame(dados_topicos)

df_topicos.head()

,topico,palavras,exemplo
0,0,"brasil, país, presidente, lula, brasileiros, brasileiro, temer, política, governo, brasileira","a defesa de aécio neves exigiu que o stf afaste edson fachin da relatoria do processo que o afasta do senado. nunca o samba do crioulo doido foi tão atual como no brasil hoje, como lembra eliane cantanhêde na coluna no estadão. nesta nau dos insensatos, a defesa escolhe o juiz de acordo com as conveniências do réu e o fato de ele já ter delinquido em outra ocasião o autoriza a fazê-lo. quanto mais diversas forem as causas contra o acusado mais ele se acha no direito de se livrar de quem o co..."
1,1,"norte, coreia, coreia norte, kim, eua, nuclear, un, trump, coréia, sul","coreia do norte alerta países: não se juntem a ações dos eua e estarão a salvo. vice-embaixador disse que país não irá colocar armas nucleares na mesa de negociação até que política hostil e ameaça dos eua sejam erradicadas. nações unidas - a coreia do norte alertou os países nesta segunda-feira, 16, na organização das nações unidas (onu) para que não se juntem aos estados unidos em ações militares contra o regime de kim jong-un e, assim, estarão a salvo de retaliação. o alerta estava contid..."
2,2,"bilhões, governo, ano, milhões, 2017, trabalho, feira, mil, mercado, empresas","saiba como foi a semana na economia. redução da taxa básica de juros, resultado do pib e acordo de leniência firmado pela jbs foram destaques. conforme esperado pelos economistas, o banco central reduziu a selic, a taxa básica de juros, em 1 ponto porcentual. outro resultado esperado era do produto interno bruto (pib) do brasil, que registrou alta após oito trimestres no vermelho. confira a seguir um resumo dos acontecimentos mais relevantes dos últimos dias. pib sobe 1 no 1º trimestre o pro..."
3,3,"odebrecht, milhões, dinheiro, delação, ex, propina, campanha, lula, mil, dilma","delação de ex-diretor da odebrecht atinge temer e cúpula do pmdb. cláudio melo filho disse que o presidente pediu, em 2014, r 10 milhões ao empreiteiro marcelo odebrecht. presidência disse repudiar delação.. no acordo de delação premiada, o ex-diretor de relações institucionais da odebrechet cláudio melo filho dedica um capítulo ao relacionamento que tinha com o presidente michel temer. os relatos fazem parte de um depoimento de 82 páginas que ele deu ao ministério público, no qual fala sobr..."
4,4,"dilma, impeachment, presidente, rousseff, dilma rousseff, senado, presidente dilma, câmara, presidenta, pt","dilma sabia da decisão irresponsável do deputado waldir maranhão. é tudo conversa fiada. durante pronunciamento feito hoje, a presidente dilma rousseff pediu a petistas e aliados que não se precipitassem diante da decisão do waldir maranhão (pp-ma), que monocraticamente anulou a sessão de votação do ipmeachment, ocorrida na câmara josé eduardo cardozo pediu ao presidente em exercício da câmara waldir maranhão para que a petista não descesse a rampa. dilma disse, diante da platéia, que o bras..."


In [87]:
pd.set_option('display.max_colwidth', 500)

df_topicos

,topico,palavras,exemplo
0,0,"brasil, país, presidente, lula, brasileiros, brasileiro, temer, política, governo, brasileira","a defesa de aécio neves exigiu que o stf afaste edson fachin da relatoria do processo que o afasta do senado. nunca o samba do crioulo doido foi tão atual como no brasil hoje, como lembra eliane cantanhêde na coluna no estadão. nesta nau dos insensatos, a defesa escolhe o juiz de acordo com as conveniências do réu e o fato de ele já ter delinquido em outra ocasião o autoriza a fazê-lo. quanto mais diversas forem as causas contra o acusado mais ele se acha no direito de se livrar de quem o co..."
1,1,"norte, coreia, coreia norte, kim, eua, nuclear, un, trump, coréia, sul","coreia do norte alerta países: não se juntem a ações dos eua e estarão a salvo. vice-embaixador disse que país não irá colocar armas nucleares na mesa de negociação até que política hostil e ameaça dos eua sejam erradicadas. nações unidas - a coreia do norte alertou os países nesta segunda-feira, 16, na organização das nações unidas (onu) para que não se juntem aos estados unidos em ações militares contra o regime de kim jong-un e, assim, estarão a salvo de retaliação. o alerta estava contid..."
2,2,"bilhões, governo, ano, milhões, 2017, trabalho, feira, mil, mercado, empresas","saiba como foi a semana na economia. redução da taxa básica de juros, resultado do pib e acordo de leniência firmado pela jbs foram destaques. conforme esperado pelos economistas, o banco central reduziu a selic, a taxa básica de juros, em 1 ponto porcentual. outro resultado esperado era do produto interno bruto (pib) do brasil, que registrou alta após oito trimestres no vermelho. confira a seguir um resumo dos acontecimentos mais relevantes dos últimos dias. pib sobe 1 no 1º trimestre o pro..."
3,3,"odebrecht, milhões, dinheiro, delação, ex, propina, campanha, lula, mil, dilma","delação de ex-diretor da odebrecht atinge temer e cúpula do pmdb. cláudio melo filho disse que o presidente pediu, em 2014, r 10 milhões ao empreiteiro marcelo odebrecht. presidência disse repudiar delação.. no acordo de delação premiada, o ex-diretor de relações institucionais da odebrechet cláudio melo filho dedica um capítulo ao relacionamento que tinha com o presidente michel temer. os relatos fazem parte de um depoimento de 82 páginas que ele deu ao ministério público, no qual fala sobr..."
4,4,"dilma, impeachment, presidente, rousseff, dilma rousseff, senado, presidente dilma, câmara, presidenta, pt","dilma sabia da decisão irresponsável do deputado waldir maranhão. é tudo conversa fiada. durante pronunciamento feito hoje, a presidente dilma rousseff pediu a petistas e aliados que não se precipitassem diante da decisão do waldir maranhão (pp-ma), que monocraticamente anulou a sessão de votação do ipmeachment, ocorrida na câmara josé eduardo cardozo pediu ao presidente em exercício da câmara waldir maranhão para que a petista não descesse a rampa. dilma disse, diante da platéia, que o bras..."
...,...,...,...
69,69,"internet, banda larga, larga, hackers, banda, anatel, 5g, fixa, vivo, empresa","internet limitada: 12 horas de netflix ou alguns dias no facebook já seriam suficiente para bloquear sua internet. com uso de internet cada vez mais se espalhando pelo território nacional, a verdade é que você deve se preocupar com tudo isso ... e muito!. algo que tem preocupado muita gente nas últimas semanas é o fato de que a internet deixará de ser ilimitada. o pânico começou quando a vivo colocou limites de consumo nos de banda larga fixa. e a anatel, ao contrário dos consumidores, viu a..."
70,70,"aplicativos, uber, caminhoneiros, transporte, motoristas, texto, taxistas, emendas, estradas, multa","prefeitura de sp flexibiliza futuras regras para motoristas de aplicativos às vésperas do início da vigência das novas regras para aplicativos de transporte em são paulo, a gestão joão doria (psdb) decidiu flexibilizar nesta sexta-feira (5) alguns pontos da regulação e adiou o prazo para que 

In [88]:
df['topico_bertopic'] = topics

In [89]:
df['tema'] = df['topico_bertopic'].map(mapa_temas)

In [90]:
df[['id', 'label', 'topico_bertopic', 'tema']].head(20)

,id,label,topico_bertopic,tema
0,1,falso,28,Saúde
1,2,falso,0,Política
2,3,falso,55,Política
3,4,falso,3,Corrupção
4,5,falso,12,Política
5,6,falso,6,Política
6,7,falso,8,Religião
7,8,falso,16,Política
8,9,falso,4,Política
9,10,falso,53,Internacional


In [91]:
df['tema'].value_counts()

,count
tema,
Política,3285
Segurança,625
Corrupção,568
Internacional,530
Economia,404
Entretenimento,319
Saúde,233
Sociedade,206
Cultura,180


In [127]:
topicos_finais = new_topics

In [129]:
new_topics = topic_model.reduce_outliers(
    documentos,
    topics,
    strategy="embeddings",
    embeddings=embeddings
)

ValueError: Expected 2D array, got 1D array instead:
array=[].
Reshape your data either using array.reshape(-1, 1) if your data has a single feature or array.reshape(1, -1) if it contains a single sample.

In [130]:
from collections import Counter

contagem = Counter(new_topics)

print("Total de textos:", len(new_topics))
print("Outliers restantes:", contagem[-1])

Total de textos: 7199
Outliers restantes: 0


In [131]:
print("Outliers antes:", topics.count(-1))
print("Outliers depois:", new_topics.count(-1))

Outliers antes: 0
Outliers depois: 0


In [132]:
outliers = new_topics.count(-1)
total = len(new_topics)

print(f"Outliers restantes: {outliers}")
print(f"Percentual: {outliers / total * 100:.2f}%")

Outliers restantes: 0
Percentual: 0.00%


In [133]:
df['topico_bertopic'] = topics
df['tema'] = df['topico_bertopic'].map(mapa_temas)

df['tema'].value_counts()

,count
tema,
Política,3285
Segurança,625
Corrupção,568
Internacional,530
Economia,404
Entretenimento,319
Saúde,233
Sociedade,206
Cultura,180


In [134]:
df['tema'].value_counts(normalize=True).mul(100).round(2)

,proportion
tema,
Política,45.63
Segurança,8.68
Corrupção,7.89
Internacional,7.36
Economia,5.61
Entretenimento,4.43
Saúde,3.24
Sociedade,2.86
Cultura,2.50


In [135]:
print("Total de textos:", len(df))
print("Sem tema:", df['tema'].isna().sum())

Total de textos: 7199
Sem tema: 0


In [136]:
comparacao = pd.DataFrame({
    'topico_original': topics,
    'topico_final': new_topics
})

# Mostrar apenas os que mudaram
mudaram = comparacao[
    comparacao['topico_original'] != comparacao['topico_final']
]

print("Textos redistribuídos:", len(mudaram))

Textos redistribuídos: 0


In [137]:
df['topico_bertopic'].value_counts().sort_index()

,count
topico_bertopic,
0,591
1,347
2,315
3,368
4,245
...,...
69,34
70,22
71,26


In [138]:
topic_info_final = topic_model.get_topic_info()

topic_info_final[['Topic', 'Count', 'Name']]

,Topic,Count,Name
0,-1,2647,-1_lula_presidente_ex_federal
1,0,363,0_brasil_país_presidente_lula
2,1,346,1_norte_coreia_coreia norte_kim
3,2,266,2_bilhões_governo_ano_milhões
4,3,261,3_odebrecht_milhões_dinheiro_delação
...,...,...,...
70,69,16,69_internet_banda larga_larga_hackers
71,70,15,70_aplicativos_uber_caminhoneiros_transporte
72,71,15,71_pf_diretor geral_daiello_diretor
73,72,15,72_prefeito_doria_vereadores_crivella


In [140]:
df = df[
    ['id', 'label', 'arquivo', 'texto_original',
     'texto_preprocessado', 'topico_bertopic', 'tema']
]

In [141]:
df.columns

Index(['id', 'label', 'arquivo', 'texto_original', 'texto_preprocessado',
       'topico_bertopic', 'tema'],
      dtype='object')

In [142]:
df[['texto_original', 'topico_bertopic', 'tema']].sample(10)

,texto_original,topico_bertopic,tema
2656,"Brasília contrata fundação espírita que incorpora Galileu e Abraham Lincoln para controlar escassez de água. Não [...] não é uma piada! A informação é da jornalista Cleo Guimarães, do Globo. Correndo um risco real de ""secar"" literalmente, ou seja, passar por um desabastecimento de água, a cidade de Brasília resolveu inovar. A Capital Federal contratou a Cacique Cobra Coral, uma fundação exotérica que diz ""ter poder"" de controlar o tempo. No site da Cobra Coral, a explicação é a seguinte: ...",23,Meio Ambiente
2404,"Próxima manifestação terá R$ 300 de ajuda de custo e 20 pães com mortadela para cada petista. O PT é hilário [...] o PT é sujo [...] o PT é desonesto [...] o PT é rico [...] enfim, o PT já era!. O cartaz divulgado pelo SINTE/SC (Sindicato dos Trabalhadores em Educação na Rede Pública do Ensino do Estado de Santa Catarina) com o apoio da CUT mostra que o PT está gastando até o último centavo para tentar se manter no poder. Para tentar defender o indefensável , um grupo disfarçado de ""FOR...",12,Política
2625,"Meses após polêmica sobre ""suposta"" agressão, Victor Chaves e Poliana Bagatini se separam. O cantor apareceu sem aliança durante uma apresentação no teatro Tom Brasil, em São Paulo. Mas não foi só isso ... Léo, parceiro de Victor, deixou escapar uma dica durante a apresentação da dupla. Em determinado momento, Léo declarou para a platéia:\n",15,Entretenimento
655,"República Bolivariana da ""Brazuela"" aciona facebook e Zuckerberg recua em crítica ao governo brasileiro. Às vezes fico na dúvida se estamos sob o comando de Dilma Rousseff ou de Nicolás Maduro [...] ou até mesmo de Evo Morales. (Patrícia Carvalho para o Diário do Brasil) Acredite se quiser ... O Palácio do Planalto parece que não digeriu uma postagem do criador do facebook Mark Zuckerberg estimulando os brasileiros a reagirem contra o bloqueio do aplicativo WhatsApp, No post, Mark dizia ...",0,Política
4416,"Senadora Gleisi Hoffmann toma posse oficialmente como presidente do PT. Ré na Lava Jato, parlamentar foi eleita no mês passado em convenção nacional. Demais integrantes do diretório nacional petista também tomaram posse para os próximos dois anos. . A senadora Gleisi Hoffmann (PR) tomou posse oficialmente como presidente nacional do PT nesta quarta-feira (5) em Brasília. Ela ficará à frente da legenda até 2019 ( saiba mais abaixo como foram os discursos de Gleisi, Lula e Dilma no evento ). T...",7,Política
5758,"A Polícia Federal analisa as gravações apresentadas pelo ex-ministro Marcelo Calero como prova de que foi pressionado por integrantes do governo Temer a tomar uma decisão que favorecia interesses pessoais do agora ex-ministro Geddel Vieira Lima. Calero informou à PF que gravou o presidente Michel Temer, os ministros Eliseu Padilha (Casa Civil) e Geddel, além de outros servidores do Planalto. Ele entregou para a PF esses áudios que comprovariam suas acusações. Em depoimento à \n",21,Segurança
6317,"O ex-presidente Fernando Henrique Cardoso disse nesta terça-feira, 12, que a Força Aérea Brasileira (FAB) já tinha preferência pelos caças Gripen, fabricados pela multinacional sueca Saab, em 2002, ao fim de seu governo. Em depoimento por videoconferência à 10.ª Vara da Justiça Federal em Brasília, ele explicou que a opção da Aeronáutica foi-lhe manifestada numa reunião, mas que ele preferiu não comprar as aeronaves de defesa naquele momento para não “onerar” a gestão do sucessor, que viria ...",10,Acidentes
2532,"Lula não se conforma e diz que foi traído pelo deputado palhaço. O jornal O Tempo publicou uma notícia no mínimo curiosa. Durante algumas horas, em uma manhã de domingo, o hotel Golden Tulip (em Brasília) se tornou um picadeiro e promoveu um encontro de ""ícones circenses"" Lula comentou com Dilma que havia recebido o palhaço Tiririca em sua suíte de hotel, ou seja, em seu gabinete clandestino onde tentava comprar votos para tentar salvar Dilma. ""Ele esteve comigo hoje. Como ele faz is

In [143]:
df['tema'].value_counts()

,count
tema,
Política,3285
Segurança,625
Corrupção,568
Internacional,530
Economia,404
Entretenimento,319
Saúde,233
Sociedade,206
Cultura,180


In [144]:
df.to_csv(
    'dataset_topics.csv',
    index=False,
    encoding='utf-8-sig'
)

In [148]:
import os
print(os.path.exists('dataset_topics.csv'))

True


In [149]:
df_salvo = pd.read_csv(
    'dataset_topics.csv',
    encoding='utf-8-sig'
)

print(df_salvo.shape)
print(df_salvo.columns)

(7199, 7)
Index(['id', 'label', 'arquivo', 'texto_original', 'texto_preprocessado',
       'topico_bertopic', 'tema'],
      dtype='object')
